# Preparar los SCF de Quantum ESPRESSO (MN5-GPP) para las configuraciones elegidas

Lee `seleccion_fps.extxyz` (la salida de `seleccion_fps_postrotura.ipynb`) y crea **una carpeta por
configuración** en `casos/`:

```
casos/case_001/estructura.extxyz   la configuración, con su información (trayectoria, t, T, fps_rank...)
casos/case_001/input.in            SCF de QE
casos/case_001/input_reintento.in  el mismo SCF con una mezcla más suave (solo si el primero no converge)
casos/indice.csv                   una fila por caso
casos/parametros_qe.json           los ajustes de QE usados
```

Opcionalmente añade unos **casos de control** (`ctrl_01`, ...): configuraciones de tu dataset de
entrenamiento que se recalculan con estos mismos ajustes. Al recoger, se compara la energía y las
fuerzas con las que ya tenías: si coinciden, los datos nuevos son coherentes con el train.

**Uso:** deja este cuaderno en la carpeta del paquete (junto a `pseudos/`, `job_scf.sh` y `workflow.py`),
copia ahí `seleccion_fps.extxyz`, revisa la sección 1 y ejecuta todo (*Run All*). Después sube la carpeta
entera a MN5 y sigue el `LEEME.md`.

**Ajustes de QE por defecto = los de tu AIMD** (`input_1.in`: ecutrho 540, smearing gaussiano de
0.005 Ry), que es de donde sale la mayor parte del train (AIMD y rattle). El `input.in` de SCF que
pasaste usa ecutrho 600 y Marzari–Vanderbilt con 0.02 Ry: con otro smearing la energía (y algo las
fuerzas) no es la misma superficie que la del train, y mezclarlas mete un error sistemático en el
reentrenamiento. Si decides usar los del SCF, cambia `ecutrho`, `smearing` y `degauss` en la sección 1.

## 0. Importaciones

In [ ]:
import csv
import json
import random
from pathlib import Path

import numpy as np
from ase.io import read, write

## 1. Parámetros

In [ ]:
# --- Entrada / salida ------------------------------------------------------------
XYZ_SELECCION = "seleccion_postrotura_new/seleccion_fps.extxyz"    # configuraciones elegidas (seleccion_fps_postrotura.ipynb)
CARPETA_CASOS = "casos"
PREFIJO = "case"                          # case_001, case_002, ...
SOBRESCRIBIR = False                      # True: rehace casos/ aunque exista (¡borra outputs!)

# --- Casos de control (opcional) ---------------------------------------------------
XYZ_CONTROL = None                        # p. ej. "train_v32_filtrado.xyz"; None = sin control
N_CONTROL = 3
FILTRO_CONTROL = "AIMD"                   # solo configuraciones cuyo config_type contenga esto
CLAVE_E_REF = "energy_qe"                 # claves de energía y fuerzas en ese fichero
CLAVE_F_REF = "forces_qe"
SEMILLA = 42

# --- Quantum ESPRESSO ----------------------------------------------------------------------
PSEUDO_DIR = "../../pseudos"              # relativo a casos/case_XXX/ (la carpeta pseudos/ del paquete)
ESPECIES = {                              # elemento: (masa, pseudopotencial), en este orden
    "C": (12.011, "C.pbe-n-kjpaw_psl.1.0.0.UPF"),
    "B": (10.811, "B.pbe-n-kjpaw_psl.1.0.0.UPF"),
}
QE = dict(
    ecutwfc=60.0,
    ecutrho=540.0,                        # AIMD y rattle: 540 (strain y tu input.in de SCF: 600)
    smearing="gaussian",                  # AIMD, rattle y strain: gaussiano (tu input.in de SCF: 'mv')
    degauss=0.005,                        # AIMD, rattle y strain: 0.005 Ry (tu input.in de SCF: 0.02)
    starting_magnetization=0.1,           # nspin = 2, como en todo el dataset
    vdw_corr="Grimme-D3",
    conv_thr=7.35e-8,                     # el de tu input.in de SCF (más estricto que el de la AIMD)
    electron_maxstep=200,
    mixing_mode="plain",
    mixing_beta=0.2,
)
# Reintento: solo cambia la mezcla (no la física), así que el resultado convergido es el mismo
QE_REINTENTO = dict(mixing_mode="local-TF", mixing_beta=0.1, mixing_ndim=12, electron_maxstep=400)

## 2. Funciones

In [ ]:
def bloque(nombre, pares):
    lineas = [f"&{nombre}"]
    for k, v in pares:
        if isinstance(v, bool):
            v = ".true." if v else ".false."
        elif isinstance(v, str):
            v = f"'{v}'"
        elif isinstance(v, float):
            v = f"{v:.10g}" if abs(v) >= 1e-3 else f"{v:.3e}"
        lineas.append(f"   {k:<26s}= {v}")
    lineas.append("/")
    return "\n".join(lineas)


def input_qe(atoms, prefijo, ajustes):
    """Texto del input de pw.x para una configuración."""
    simbolos = atoms.get_chemical_symbols()
    falta = sorted(set(simbolos) - set(ESPECIES))
    if falta:
        raise ValueError(f"Elementos sin pseudopotencial: {falta}")
    especies = [e for e in ESPECIES if e in simbolos]
    control = [("calculation", "scf"), ("restart_mode", "from_scratch"), ("prefix", prefijo),
               ("outdir", "./tmp"), ("pseudo_dir", PSEUDO_DIR), ("disk_io", "nowf"),
               ("tstress", True), ("tprnfor", True)]
    system = [("ibrav", 0), ("nat", len(atoms)), ("ntyp", len(especies)),
              ("ecutwfc", ajustes["ecutwfc"]), ("ecutrho", ajustes["ecutrho"]), ("nosym", True),
              ("occupations", "smearing"), ("smearing", ajustes["smearing"]),
              ("degauss", ajustes["degauss"]), ("nspin", 2)]
    system += [(f"starting_magnetization({i + 1})", ajustes["starting_magnetization"])
               for i in range(len(especies))]
    system += [("vdw_corr", ajustes["vdw_corr"])]
    electrons = [("electron_maxstep", ajustes["electron_maxstep"]), ("conv_thr", ajustes["conv_thr"]),
                 ("mixing_mode", ajustes["mixing_mode"]), ("mixing_beta", ajustes["mixing_beta"])]
    if "mixing_ndim" in ajustes:
        electrons.append(("mixing_ndim", ajustes["mixing_ndim"]))
    partes = [bloque("CONTROL", control), bloque("SYSTEM", system), bloque("ELECTRONS", electrons), ""]
    partes.append("ATOMIC_SPECIES")
    partes += [f"{e:<2s} {ESPECIES[e][0]:9.4f}  {ESPECIES[e][1]}" for e in especies]
    partes += ["", "K_POINTS gamma", "", "CELL_PARAMETERS angstrom"]
    partes += ["  " + "  ".join(f"{x:18.12f}" for x in v) for v in atoms.cell.array]
    partes += ["", "ATOMIC_POSITIONS angstrom"]
    partes += [f"{s:<2s} " + "  ".join(f"{x:18.12f}" for x in p)
               for s, p in zip(simbolos, atoms.positions)]
    return "\n".join(partes) + "\n"


def d_min(atoms):
    d = atoms.get_all_distances(mic=True)
    np.fill_diagonal(d, np.inf)
    return float(d.min())


def preparar_caso(nombre, atoms, info_extra, referencia=None):
    carpeta = Path(CARPETA_CASOS) / nombre
    carpeta.mkdir(parents=True, exist_ok=False)
    a = atoms.copy()
    a.calc = None
    for clave in list(a.arrays):
        if clave not in ("numbers", "positions"):
            del a.arrays[clave]
    a.pbc = True
    a.info = {k: v for k, v in atoms.info.items() if np.isscalar(v)}
    a.info.update(info_extra)
    write(carpeta / "estructura.extxyz", a, format="extxyz")
    (carpeta / "input.in").write_text(input_qe(a, nombre, QE))
    (carpeta / "input_reintento.in").write_text(input_qe(a, nombre, {**QE, **QE_REINTENTO}))
    if referencia is not None:
        (carpeta / "referencia.json").write_text(json.dumps(referencia))
    return a

## 3. Leer la selección (y el control)

In [ ]:
seleccion = read(XYZ_SELECCION, index=":")
print(f"{XYZ_SELECCION}: {len(seleccion)} configuraciones, "
      f"{sorted(set(len(a) for a in seleccion))} átomos, elementos {sorted(set(a.get_chemical_formula() for a in seleccion))}")

control = []
if XYZ_CONTROL:
    todas = read(XYZ_CONTROL, index=":")
    aptas = [a for a in todas if FILTRO_CONTROL in str(a.info.get("config_type", ""))
             and CLAVE_E_REF in a.info and CLAVE_F_REF in a.arrays]
    if len(aptas) < N_CONTROL:
        raise ValueError(f"Solo {len(aptas)} configuraciones de control con '{FILTRO_CONTROL}', "
                         f"{CLAVE_E_REF} y {CLAVE_F_REF} en {XYZ_CONTROL}.")
    control = random.Random(SEMILLA).sample(aptas, N_CONTROL)
    print(f"{XYZ_CONTROL}: {len(todas)} configuraciones, {len(aptas)} aptas; {N_CONTROL} de control "
          f"({', '.join(str(a.info.get('config_type')) for a in control)})")

## 4. Crear las carpetas

In [ ]:
raiz = Path(CARPETA_CASOS)
if raiz.exists():
    hay_outputs = any(raiz.glob("*/output*.out"))
    if not SOBRESCRIBIR:
        raise FileExistsError(f"{raiz}/ ya existe{' y tiene outputs' if hay_outputs else ''}. "
                              "Bórrala o pon SOBRESCRIBIR = True.")
    import shutil
    shutil.rmtree(raiz)
raiz.mkdir()

pseudo_abs = (raiz / "case_000" / PSEUDO_DIR).resolve()
for e, (_, upf) in ESPECIES.items():
    if not (pseudo_abs / upf).exists():
        print(f"  aviso: no encuentro {pseudo_abs / upf} (pon los .UPF en esa carpeta o cambia PSEUDO_DIR)")

COLUMNAS = ["caso", "tipo", "n_atomos", "formula", "a_A", "c_A", "d_min_A", "config_type", "traj",
            "frame", "t_ps", "T_obj_K", "T_inst_K", "BC_intactos", "area_A2_at", "z_rms_A",
            "espesor_A", "ps_antes_colapso", "fps_rank", "E_ref_eV_at"]
filas = []
ancho = max(3, len(str(len(seleccion))))
for i, at in enumerate(seleccion, start=1):
    nombre = f"{PREFIJO}_{i:0{ancho}d}"
    a = preparar_caso(nombre, at, {"caso": nombre})
    fila = {k: at.info[k] for k in COLUMNAS if k in at.info}          # traj, frame, t_ps, fps_rank...
    fila.update(caso=nombre, tipo="postrotura", n_atomos=len(a), formula=a.get_chemical_formula(),
                a_A=round(float(np.linalg.norm(a.cell[0])), 4), c_A=round(float(a.cell[2, 2]), 3),
                d_min_A=round(d_min(a), 4))
    filas.append(fila)
for j, at in enumerate(control, start=1):
    nombre = f"ctrl_{j:02d}"
    n = len(at)
    ref = dict(energy_eV=float(at.info[CLAVE_E_REF]), forces_eV_A=np.asarray(at.arrays[CLAVE_F_REF]).tolist(),
               fichero=str(XYZ_CONTROL), config_type=str(at.info.get("config_type", "")))
    a = preparar_caso(nombre, at, {"caso": nombre, "tipo": "control"}, ref)
    filas.append(dict(caso=nombre, tipo="control", n_atomos=n, formula=a.get_chemical_formula(),
                      a_A=round(float(np.linalg.norm(a.cell[0])), 4), c_A=round(float(a.cell[2, 2]), 3),
                      d_min_A=round(d_min(a), 4), config_type=ref["config_type"],
                      E_ref_eV_at=ref["energy_eV"] / n))

with open(raiz / "indice.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=COLUMNAS, extrasaction="ignore")
    w.writeheader()
    for fila in filas:
        w.writerow(fila)
(raiz / "parametros_qe.json").write_text(json.dumps(
    dict(QE=QE, QE_REINTENTO=QE_REINTENTO, PSEUDO_DIR=PSEUDO_DIR,
         ESPECIES={k: list(v) for k, v in ESPECIES.items()}, XYZ_SELECCION=XYZ_SELECCION,
         XYZ_CONTROL=XYZ_CONTROL), indent=2))

print(f"\nCreados {len(filas)} casos en {raiz}/ ({len(seleccion)} de la selección"
      f"{f', {len(control)} de control' if control else ''})")
print(f"  c = {sorted(set(f['c_A'] for f in filas))} Å; a = {min(f['a_A'] for f in filas):.2f}–"
      f"{max(f['a_A'] for f in filas):.2f} Å; d_min ≥ {min(f['d_min_A'] for f in filas):.3f} Å")

## 5. Comprobación: QE leerá lo mismo que hay en estructura.extxyz

In [ ]:
# Se relee cada input.in con el lector de ASE para pw.x y se compara con la configuración.
max_dpos, max_dcel, n_ok = 0.0, 0.0, 0
for fila in filas:
    carpeta = raiz / fila["caso"]
    ref = read(carpeta / "estructura.extxyz")
    for nombre_in in ("input.in", "input_reintento.in"):
        qe = read(carpeta / nombre_in, format="espresso-in")
        assert qe.get_chemical_symbols() == ref.get_chemical_symbols(), fila["caso"]
        max_dpos = max(max_dpos, float(np.abs(qe.positions - ref.positions).max()))
        max_dcel = max(max_dcel, float(np.abs(qe.cell.array - ref.cell.array).max()))
    n_ok += 1
print(f"{n_ok} casos comprobados: |Δposición| máx = {max_dpos:.1e} Å, |Δcelda| máx = {max_dcel:.1e} Å")
print("\nPrimer input.in:\n")
print((raiz / filas[0]["caso"] / "input.in").read_text()[:1500], "...")